# MoE Intuition Walkthrough (Ref Kernel)

This notebook explains the hardest parts of `ref_moe.py` with tiny tensors:

1. Grouping and group-score routing intuition
2. Why `unsqueeze`, `repeat`, and `repeat_interleave` are needed
3. What happens if we skip those operations
4. What the local-expert `for` loop is doing

Goal: make every shape transform and routing decision visible.

In [1]:
import torch

torch.set_printoptions(precision=4, sci_mode=False, linewidth=140)
print('Torch:', torch.__version__)
device = 'cpu'

Torch: 2.9.1+cu128


## 1) Grouping + group scores (intuitive view)

Think of experts as being arranged in groups.

- We first find **strong groups** (not individual experts yet).
- Group score = sum of top-2 experts in that group.
- Keep only top `TOPK_GROUP` groups.
- Then pick final `TOP_K` experts globally, but only from kept groups.

This avoids sending tokens across too many groups and stabilizes routing.

In [2]:
# Tiny routing setup so we can inspect everything by eye
T = 2
E_global = 16
N_GROUP = 4
group_size = E_global // N_GROUP
TOPK_GROUP = 2
TOP_K = 4

routing_logits = torch.tensor([
    [ 1.2, -0.5,  0.1,  0.3,   2.0,  1.8, -1.0,  0.2,   0.4,  0.5, -0.2, -0.1,   1.5,  1.4,  0.0, -0.6],
    [-0.3,  0.8,  1.1, -0.9,   0.2, -0.4,  1.6,  1.7,   1.0,  0.9,  0.1, -0.8,  -0.2,  0.4,  1.3,  1.2],
], dtype=torch.float32)

routing_bias = torch.tensor([0.0, 0.05, 0.0, 0.0,   0.02, 0.02, 0.0, 0.0,   0.01, 0.0, 0.0, 0.0,   0.03, 0.0, 0.0, 0.0], dtype=torch.float32)

s = torch.sigmoid(routing_logits)
s_with_bias = s + routing_bias

print('s shape:', tuple(s.shape))
print('s_with_bias shape:', tuple(s_with_bias.shape))

s_grouped = s_with_bias.view(T, N_GROUP, group_size)
print('\nGrouped scores shape [T, N_GROUP, group_size]:', tuple(s_grouped.shape))
print('Token0 grouped view:\n', s_grouped[0])

top2_vals, top2_idx = torch.topk(s_grouped, k=2, dim=2, largest=True, sorted=True)
group_scores = top2_vals.sum(dim=2)

print('\nTop2 per group (token0):\n', top2_vals[0])
print('Group scores [T, N_GROUP]:\n', group_scores)

group_keep = torch.topk(group_scores, k=TOPK_GROUP, dim=1, largest=True, sorted=True).indices
group_mask = torch.zeros_like(group_scores, dtype=torch.bool)
group_mask.scatter_(1, group_keep, True)

print('\nSelected groups indices [T, TOPK_GROUP]:\n', group_keep)
print('Group mask [T, N_GROUP]:\n', group_mask)

score_mask = group_mask.unsqueeze(2).expand(T, N_GROUP, group_size).reshape(T, E_global)
scores_pruned = s_with_bias.masked_fill(~score_mask, float('-inf'))
topk_idx = torch.topk(scores_pruned, k=TOP_K, dim=1, largest=True, sorted=True).indices

print('\nScore mask [T, E_global]:\n', score_mask)
print('Final topk_idx [T, TOP_K]:\n', topk_idx)

s shape: (2, 16)
s_with_bias shape: (2, 16)

Grouped scores shape [T, N_GROUP, group_size]: (2, 4, 4)
Token0 grouped view:
 tensor([[0.7685, 0.4275, 0.5250, 0.5744],
        [0.9008, 0.8781, 0.2689, 0.5498],
        [0.6087, 0.6225, 0.4502, 0.4750],
        [0.8476, 0.8022, 0.5000, 0.3543]])

Top2 per group (token0):
 tensor([[0.7685, 0.5744],
        [0.9008, 0.8781],
        [0.6225, 0.6087],
        [0.8476, 0.8022]])
Group scores [T, N_GROUP]:
 tensor([[1.3430, 1.7789, 1.2311, 1.6498],
        [1.4902, 1.6776, 1.4520, 1.5544]])

Selected groups indices [T, TOPK_GROUP]:
 tensor([[1, 3],
        [1, 3]])
Group mask [T, N_GROUP]:
 tensor([[False,  True, False,  True],
        [False,  True, False,  True]])

Score mask [T, E_global]:
 tensor([[False, False, False, False,  True,  True,  True,  True, False, False, False, False,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True, False, False, False, False,  True,  True,  True,  True]])
Final topk

### Why this is done in 2 stages

- Stage A (group-level): "Which neighborhoods look promising?"
- Stage B (expert-level): "Within promising neighborhoods, which exact experts win?"

This avoids a pure flat top-k over all experts and gives better locality/control.

## 2) `unsqueeze` + `repeat` for hidden-state scales

Reference pattern:

- hidden states: `[T, H]`
- scales: `[H/128, T]` then permuted to `[T, H/128]`
- each scale applies to 128 hidden dims

`unsqueeze(-1)` makes `[T, H/128, 1]` so we can repeat along the 128-block axis.

In [ ]:
T = 2
H = 8
BLOCK = 2
num_blocks = H // BLOCK

hidden_states = torch.arange(T * H, dtype=torch.float32).reshape(T, H)
# scale layout in ref: [H/BLOCK, T]
hidden_states_scale = torch.tensor([
    [1.0, 10.0],  # block0 scale for token0/token1
    [2.0, 20.0],
    [3.0, 30.0],
    [4.0, 40.0],
], dtype=torch.float32)

scale_TH = hidden_states_scale.permute(1, 0).contiguous()     # [T, H/BLOCK]
expanded = scale_TH.unsqueeze(-1).repeat(1, 1, BLOCK).reshape(T, H)
A = hidden_states * expanded

print('hidden_states:\n', hidden_states)
print('\nscale_TH [T, H/BLOCK]:\n', scale_TH)
print('\nexpanded scale [T, H]:\n', expanded)
print('\nA = hidden * expanded:\n', A)

# Verify with direct formula h//BLOCK
A_formula = torch.empty_like(A)
for t in range(T):
    for h in range(H):
        A_formula[t, h] = hidden_states[t, h] * hidden_states_scale[h // BLOCK, t]

print('\nMatches direct formula:', torch.allclose(A, A_formula))

If you **do not** expand scales to `[T, H]`, the multiply is semantically wrong (or shape-incompatible).

The block-scale tensor stores one scale per block, not per element. Expansion applies each block scale to all elements in that block.

## 3) `repeat_interleave` for weight scales

Reference uses:

```python
S13_expanded = torch.repeat_interleave(S13, BLOCK, dim=1)
S13_expanded = torch.repeat_interleave(S13_expanded, BLOCK, dim=2)
W13 = W13_fp32 * S13_expanded
```

Meaning: scale is stored per `(out_block, in_block)`, then expanded so every element in that block gets the same scale.

In [ ]:
E = 1
BLOCK = 2
out_dim = 4
in_dim = 6

# fake fp8 already cast to fp32 for demo
W = torch.arange(E * out_dim * in_dim, dtype=torch.float32).reshape(E, out_dim, in_dim)

# block scales: [E, out_dim/BLOCK, in_dim/BLOCK] = [1,2,3]
S = torch.tensor([[[1.0, 2.0, 3.0],
                   [10.0, 20.0, 30.0]]], dtype=torch.float32)

S_exp = torch.repeat_interleave(S, BLOCK, dim=1)
S_exp = torch.repeat_interleave(S_exp, BLOCK, dim=2)
W_scaled = W * S_exp

print('W shape:', tuple(W.shape))
print('S shape:', tuple(S.shape))
print('S_exp shape:', tuple(S_exp.shape))
print('\nS_exp[0]:\n', S_exp[0])
print('\nW_scaled[0]:\n', W_scaled[0])

# Check one element against block formula
o, i = 3, 4
lhs = W_scaled[0, o, i].item()
rhs = W[0, o, i].item() * S[0, o // BLOCK, i // BLOCK].item()
print('\nCheck element (o=3,i=4):', lhs, '==', rhs)

If we skip `repeat_interleave`, we cannot correctly apply block scales to per-element weights. Either shapes do not match, or we accidentally apply wrong broadcasting semantics.

## 4) Combination weights: why use `s`, not `s_with_bias`

Routing bias helps *selection* of experts. But final mixing weights are normalized from raw `s` values for selected experts.

In [ ]:
TOP_K = 4
s = torch.tensor([[0.2, 0.5, 0.9, 0.1, 0.7]], dtype=torch.float32)
s_with_bias = s + torch.tensor([[0.0, 0.0, -0.3, 0.0, 0.2]], dtype=torch.float32)
topk_idx = torch.topk(s_with_bias, k=TOP_K, dim=1).indices

M = torch.zeros_like(s)
M.scatter_(1, topk_idx, 1.0)
weights = s * M
weights = weights / (weights.sum(dim=1, keepdim=True) + 1e-20)

print('s:', s)
print('s_with_bias:', s_with_bias)
print('topk_idx from s_with_bias:', topk_idx)
print('normalized weights from s:', weights)

## 5) What the local-expert `for` loop is doing

Loop idea:

- Iterate local experts `le`
- Map to global expert `ge = local_offset + le`
- Find tokens that selected `ge`
- Run expert MLP only on those tokens
- Multiply by token-specific route weights
- Accumulate into output rows

In [ ]:
# Tiny simulation of only the loop logic (no real GEMM)
T = 5
E_global = 8
E_local = 2
TOP_K = 3
H = 4
local_expert_offset = 3  # local experts correspond to global experts 3 and 4

topk_idx = torch.tensor([
    [3, 1, 6],
    [0, 4, 7],
    [2, 5, 6],
    [4, 3, 1],
    [7, 0, 4],
], dtype=torch.long)

weights = torch.zeros(T, E_global)
weights[0, 3] = 0.6
weights[1, 4] = 0.7
weights[3, 3] = 0.3
weights[3, 4] = 0.5
weights[4, 4] = 0.4

output = torch.zeros(T, H)

for le in range(E_local):
    ge = local_expert_offset + le
    sel_mask_per_token = (topk_idx == ge).any(dim=1)
    if not sel_mask_per_token.any():
        continue

    token_idx = torch.nonzero(sel_mask_per_token, as_tuple=False).squeeze(1)
    Tk = token_idx.numel()

    # Fake expert output: every selected token gets vector [ge, ge, ge, ge]
    O = torch.full((Tk, H), float(ge))
    w_tok = weights.index_select(0, token_idx)[:, ge]
    output.index_add_(0, token_idx, O * w_tok.unsqueeze(1))

    print(f'local expert {le}, global expert {ge}')
    print('  token_idx =', token_idx.tolist())
    print('  w_tok     =', w_tok.tolist())

print('\nFinal accumulated output:\n', output)

### Why the loop is sparse and dynamic

- Each expert sees different token counts (`Tk` changes per expert).
- Some experts get zero tokens and are skipped.
- This is why efficient dispatch/gather/scatter matters for performance.

## 6) Practical summary

- `unsqueeze/repeat/reshape` on hidden scales: convert block scales to per-element scales.
- `repeat_interleave` on weight scales: same idea for weight blocks on both axes.
- Grouped routing: first rank groups, then rank experts within allowed groups.
- Loop over local experts: sparse compute only for selected tokens, then weighted accumulation.

If you want, next we can add a second notebook that mirrors your real MoE constants (`H=7168`, `I=2048`, `E=256`) and times each stage, so you can connect intuition to performance cost.